# SmolVLM 256M experiment record
Executed scoring of saved predictions, not a second training run.

Original: `V3_EdgeVLM_256M_Phase2_NoMediaPipe_EvaluationOnly_run.ipynb`. Numeric/text outputs are retained where present. Embedded visual outputs and image-path previews are removed for publication. Execution counts are not fabricated. See README.md and the source manifest.

This is a historical server notebook, not a portable CLI. Review paths, dependencies, adapter initialization and optional export cells before executing. No dataset or weights are included.


# SmolVLM-256M Phase 2 — Evaluation Only

This notebook reports metrics from the completed No-MediaPipe test run. It reads the already-saved test predictions CSV; it does **not** train the model or run GPU inference.

It verifies the saved labels against the Phase 2 test split, calculates accuracy, balanced accuracy, Macro-F1, per-class scores, confusion matrix, safety-related error rates, and latency, then saves compact report files beside the predictions CSV.

## 1. Configure paths and metric labels

Check that the paths below match the Jupyter server. The test predictions file should already exist from the completed run.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_recall_fscore_support,
)

HOME = Path('/home/jovyan')
OUTPUT_ROOT = HOME / 'EdgeVLM_finetune vlm256' / 'phase2_256m_improved_sqrt_lookawayaug_1_lr5e-05'
PREDICTIONS_CSV = OUTPUT_ROOT / 'phase2_256m_all_techniques_test_predictions.csv'
TEST_CSV = HOME / 'phase2-tain-val-test-splits' / 'final_test.csv'
LABELS = ['NORMAL', 'DROWSY', 'LOOKING_AWAY', 'PHONE_USAGE']

print('Predictions:', PREDICTIONS_CSV, '| exists:', PREDICTIONS_CSV.is_file())
print('Test split:', TEST_CSV, '| exists:', TEST_CSV.is_file())
if not PREDICTIONS_CSV.is_file():
    raise FileNotFoundError(PREDICTIONS_CSV)
if not TEST_CSV.is_file():
    raise FileNotFoundError(TEST_CSV)

Predictions: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_improved_sqrt_lookawayaug_1_lr5e-05/phase2_256m_all_techniques_test_predictions.csv | exists: True
Test split: /home/jovyan/phase2-tain-val-test-splits/final_test.csv | exists: True


## 2. Load and verify the saved test predictions

This checks that there is exactly one prediction for every test row and that the saved ground-truth labels still match the test split.

In [2]:
predictions_df = pd.read_csv(PREDICTIONS_CSV).sort_values('row_index').reset_index(drop=True)
test_df = pd.read_csv(TEST_CSV).reset_index(drop=True)
required = {'row_index', 'true_label', 'prediction', 'exact_label_valid', 'latency_ms'}
missing = required - set(predictions_df.columns)
if missing:
    raise ValueError(f'Predictions CSV is missing columns: {sorted(missing)}')
if len(predictions_df) != len(test_df):
    raise ValueError(f'Expected {len(test_df)} predictions, found {len(predictions_df)}')
if predictions_df['row_index'].duplicated().any():
    raise ValueError('Predictions CSV contains duplicate test row indexes.')
if predictions_df['row_index'].astype(int).tolist() != list(range(len(test_df))):
    raise ValueError('Predictions do not cover the test split in row-index order.')

y_true = predictions_df['true_label'].astype(str).str.strip().str.upper().to_numpy()
y_pred = predictions_df['prediction'].astype(str).str.strip().str.upper().to_numpy()
expected_true = test_df['label'].astype(str).str.strip().str.upper().to_numpy()
if not np.array_equal(y_true, expected_true):
    raise ValueError('Saved true labels do not match final_test.csv; refusing to report mismatched metrics.')
unknown_true = sorted(set(y_true) - set(LABELS))
unknown_pred = sorted(set(y_pred) - set(LABELS) - {'INVALID'})
if unknown_true or unknown_pred:
    raise ValueError(f'Unexpected labels: true={unknown_true}, predicted={unknown_pred}')

print(f'Validated {len(predictions_df):,} predictions against the test split.')
print('Prediction labels:', predictions_df['prediction'].value_counts().to_dict())

Validated 5,558 predictions against the test split.
Prediction labels: {'NORMAL': 3203, 'DROWSY': 1201, 'PHONE_USAGE': 731, 'LOOKING_AWAY': 423}


## 3. Calculate final test metrics

The test set is used only for final reporting here; do not use these results to tune training settings.

In [3]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=LABELS, zero_division=0
)
matrix_labels = LABELS + ['INVALID']
cm = confusion_matrix(y_true, y_pred, labels=matrix_labels)
latencies = pd.to_numeric(predictions_df['latency_ms'], errors='coerce').dropna().to_numpy()

unsafe_true = np.isin(y_true, ['DROWSY', 'LOOKING_AWAY', 'PHONE_USAGE'])
normal_true = y_true == 'NORMAL'
unsafe_pred = np.isin(y_pred, ['DROWSY', 'LOOKING_AWAY', 'PHONE_USAGE'])
unsafe_to_safe_miss = float(np.mean(np.isin(y_pred[unsafe_true], ['NORMAL', 'INVALID'])))
unsafe_class_error = float(np.mean(y_pred[unsafe_true] != y_true[unsafe_true]))
false_alarm = float(np.mean(unsafe_pred[normal_true]))
validity_text = predictions_df['exact_label_valid'].astype(str).str.strip().str.lower()
exact_validity = validity_text.isin(['true', '1', 'yes']).mean()
parser_failure = float(np.mean(y_pred == 'INVALID'))

metrics = {
    'images': int(len(predictions_df)),
    'accuracy_percent': round(float(accuracy_score(y_true, y_pred) * 100), 2),
    'balanced_accuracy_percent': round(float(recall.mean() * 100), 2),
    'macro_f1_percent': round(float(f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0) * 100), 2),
    'exact_label_validity_percent': round(float(exact_validity * 100), 2),
    'parser_failure_percent': round(parser_failure * 100, 2),
    'unsafe_to_safe_miss_rate_percent': round(unsafe_to_safe_miss * 100, 2),
    'unsafe_class_error_rate_percent': round(unsafe_class_error * 100, 2),
    'false_alarm_rate_percent': round(false_alarm * 100, 2),
    'mean_latency_ms': round(float(np.mean(latencies)), 2) if len(latencies) else None,
    'p95_latency_ms': round(float(np.percentile(latencies, 95)), 2) if len(latencies) else None,
}

print('========== 256M NO-MEDIAPIPE PHASE 2 TEST METRICS ==========')
for name, value in metrics.items():
    print(f'{name}: {value}')

per_class = pd.DataFrame({
    'label': LABELS, 'precision': precision, 'recall': recall,
    'f1': f1, 'support': support.astype(int),
})
display(per_class)
print('Confusion matrix (rows=true, columns=predicted):')
display(pd.DataFrame(cm, index=matrix_labels, columns=matrix_labels))

========== 256M NO-MEDIAPIPE PHASE 2 TEST METRICS ==========
images: 5558
accuracy_percent: 85.16
balanced_accuracy_percent: 81.89
macro_f1_percent: 79.63
exact_label_validity_percent: 100.0
parser_failure_percent: 0.0
unsafe_to_safe_miss_rate_percent: 13.37
unsafe_class_error_rate_percent: 19.55
false_alarm_rate_percent: 11.57
mean_latency_ms: 436.4
p95_latency_ms: 568.83


          label  precision    recall        f1  support
0        NORMAL   0.904777  0.884345  0.894444     3277
1        DROWSY   0.738551  0.868756  0.798380     1021
2  LOOKING_AWAY   0.555556  0.778146  0.648276      302
3   PHONE_USAGE   0.975376  0.744259  0.844287      958

Confusion matrix (rows=true, columns=predicted):


              NORMAL  DROWSY  LOOKING_AWAY  PHONE_USAGE  INVALID
NORMAL          2898     281            85           13        0
DROWSY           130     887             4            0        0
LOOKING_AWAY      56       6           235            5        0
PHONE_USAGE      119      27            99          713        0
INVALID            0       0             0            0        0

## 4. Save the report files

The original predictions CSV is left unchanged. These report files are written beside it.

In [4]:
per_class.to_csv(OUTPUT_ROOT / 'evaluation_only_per_class_metrics.csv', index=False)
pd.DataFrame(cm, index=matrix_labels, columns=matrix_labels).rename_axis('true_label').to_csv(
    OUTPUT_ROOT / 'evaluation_only_confusion_matrix.csv'
)
with (OUTPUT_ROOT / 'evaluation_only_test_metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(metrics, handle, indent=2)
print('Saved:')
print(OUTPUT_ROOT / 'evaluation_only_test_metrics.json')
print(OUTPUT_ROOT / 'evaluation_only_per_class_metrics.csv')
print(OUTPUT_ROOT / 'evaluation_only_confusion_matrix.csv')

Saved:
/home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_improved_sqrt_lookawayaug_1_lr5e-05/evaluation_only_test_metrics.json
/home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_improved_sqrt_lookawayaug_1_lr5e-05/evaluation_only_per_class_metrics.csv
/home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_improved_sqrt_lookawayaug_1_lr5e-05/evaluation_only_confusion_matrix.csv
